# Tarea 4: Análisis de Datos y Optimización

## **MABEL CRIS FARFAN TIPO**

**Fecha de entrega:** [VER CANVAS]

**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa los tres problemas en este notebook
- Escribe tu código en las celdas indicadas
- Ejecuta todas las celdas para verificar que tu código funciona
- Guarda tu archivo `.ipynb` en la carpeta `tareas` de tu repositorio privado de GitHub (compartido con el docente)
- Envía el enlace a tu notebook en Canvas

**⚠️ IMPORTANTE:** GitHub registra el historial de cambios de cada archivo. Tu notebook debe ser subido a GitHub **antes del plazo**. **NO** modifiques el archivo después del plazo — los cambios tardíos serán detectados y pueden resultar en penalidad.

**Integridad académica:** Esta es una tarea individual. Puedes consultar los materiales del curso, documentación de Python, herramientas de IA y discutir conceptos con compañeros, pero todo el código debe ser tuyo.

---

In [1]:
# Importaciones estándar - ejecuta esta celda primero
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, curve_fit
from io import StringIO

---
## Problema 1: Análisis de Calidad de Agua en Ríos Andino-Amazónicos (7 puntos)

Estás analizando datos de calidad de agua de estaciones de monitoreo en tres ríos de la cuenca amazónica peruana, provenientes de la red de monitoreo de la **Autoridad Nacional del Agua (ANA)**. El conjunto de datos contiene mediciones de temperatura, oxígeno disuelto (OD), pH y conductividad eléctrica recolectadas durante varios meses.

### Tus Tareas:

**Parte A (2 puntos):** Carga y explora los datos
1. Carga los datos del string CSV provisto abajo en un DataFrame de pandas
2. Muestra información básica del conjunto de datos (forma, tipos de datos, primeras filas)
3. Verifica valores faltantes e indica cuántos hay en cada columna
4. Convierte la columna `fecha` a formato datetime usando `pd.to_datetime()`

**Parte B (3 puntos):** Análisis de datos con agrupación
1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (`od_mg_l`) agrupado por `estacion_id`
2. Determina qué estación tiene la media más baja de oxígeno disuelto
3. Crea una nueva columna llamada `estado_od` que clasifique cada medición como:
   - "Crítico" si OD < 4 mg/L
   - "Bajo" si OD está entre 4 y 6 mg/L
   - "Adecuado" si OD está entre 6 y 8 mg/L
   - "Bueno" si OD >= 8 mg/L
4. Cuenta cuántas mediciones caen en cada categoría de `estado_od` por estación

**Parte C (2 puntos):** Filtrado y resumen
1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"

In [6]:
# Dataset de calidad de agua - ríos andino-amazónicos del Perú
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)

# Parte A: Carga y explora los datos
from io import StringIO
# Pista: Usa pd.read_csv(StringIO(calidad_agua_csv))
df = pd.read_csv(StringIO(calidad_agua_csv))

print("Forma del dataset:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nPrimeras filas:")
print(df.head())

print("\nValores faltantes por columna:")
print(df.isnull().sum())

df['fecha'] = pd.to_datetime(df['fecha'])
print("\nTipo de dato de 'fecha' luego de la conversión:", df['fecha'].dtype)

Forma del dataset: (24, 6)

Tipos de datos:
estacion_id          object
fecha                object
temp_c              float64
od_mg_l             float64
ph                  float64
conductividad_us      int64
dtype: object

Primeras filas:
   estacion_id       fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI  2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI  2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI  2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI  2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI  2024-07-03    24.2      5.2  7.0               172

Valores faltantes por columna:
estacion_id         0
fecha               0
temp_c              0
od_mg_l             0
ph                  0
conductividad_us    0
dtype: int64

Tipo de dato de 'fecha' luego de la conversión: datetime64[ns]


In [9]:
# Parte B: Análisis de datos con agrupación
od_por_estacion = df.groupby('estacion_id')['od_mg_l'].agg(['mean', 'std', 'min', 'max'])
print("Estadísticas de OD por estación:")
print(od_por_estacion)

estacion_menor_od = od_por_estacion['mean'].idxmin()
print(f"\nLa estación con la media mas baja de OD es: {estacion_menor_od}")

def clasificar_od(od):
    if od < 4:
        return 'Critico'
    elif od < 6:
        return 'Bajo'
    elif od < 8:
        return 'Adecuado'
    else:
        return 'Bueno'

df['estado_od'] = df['od_mg_l'].apply(clasificar_od)

conteo_estados = df.groupby(['estacion_id', 'estado_od']).size()
print("\nConteo de mediciones por estado de OD y estación:")
print(conteo_estados)


Estadísticas de OD por estación:
                 mean       std  min  max
estacion_id                              
RIO_MANTARO    7.0875  1.709166  4.8  9.5
RIO_TAMBOPATA  7.3750  0.692305  6.5  8.5
RIO_UCAYALI    5.6875  1.062931  4.2  7.2

La estación con la media mas baja de OD es: RIO_UCAYALI

Conteo de mediciones por estado de OD y estación:
estacion_id    estado_od
RIO_MANTARO    Adecuado     2
               Bajo         3
               Bueno        3
RIO_TAMBOPATA  Adecuado     6
               Bueno        2
RIO_UCAYALI    Adecuado     3
               Bajo         5
dtype: int64


In [11]:
# Parte C: Filtrado y resumen
df_filtrado = df[(df['temp_c'] > 20) & (df['ph'] >= 6.5) & (df['ph'] <= 8.5)].copy()
print(f"Número de mediciones que cumplen el filtro: {len(df_filtrado)}")

df_filtrado['mes'] = df_filtrado['fecha'].dt.month
conductividad_mensual = df_filtrado.groupby('mes')['conductividad_us'].mean()
print("\nConductividad media por mes (datos filtrados):")
print(conductividad_mensual)

df['mes'] = df['fecha'].dt.month
df_criticos_bajos = df[df['estado_od'].isin(['Critico', 'Bajo'])]
conteo_estacion_mes = df_criticos_bajos.groupby(['estacion_id', 'mes']).size().sort_values(ascending=False)
print("\nConteo de lecturas Critico/Bajo por estación y mes:")
print(conteo_estacion_mes)

combinacion_max = conteo_estacion_mes.idxmax()
print(f"\nLa combinación con mas lecturas Critico/Bajo es: estación={combinacion_max[0]}, mes={combinacion_max[1]}, con {conteo_estacion_mes.max()} lecturas")


Número de mediciones que cumplen el filtro: 16

Conductividad media por mes (datos filtrados):
mes
5    125.00
6    138.25
7    146.50
8    146.75
Name: conductividad_us, dtype: float64

Conteo de lecturas Critico/Bajo por estación y mes:
estacion_id  mes
RIO_MANTARO  8      2
RIO_UCAYALI  8      2
             7      2
RIO_MANTARO  7      1
RIO_UCAYALI  6      1
dtype: int64

La combinación con mas lecturas Critico/Bajo es: estación=RIO_MANTARO, mes=8, con 2 lecturas


---
## Problema 2: Comparación Estadística de Parcelas Forestales en la Amazonía (6 puntos)

Investigadores del **INIA (Instituto Nacional de Innovación Agraria) - Estación Experimental Pucallpa** midieron la biomasa arbórea (kg) en parcelas pareadas — unas sometidas a un tratamiento de aprovechamiento forestal de impacto reducido (AFIR) y otras dejadas como control. Se quiere determinar si el tratamiento afectó significativamente la biomasa individual de los árboles y si existe relación entre el diámetro y la biomasa.

### Tus Tareas:

**Parte A (2 puntos):** Comparación de grupos de tratamiento
1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión

**Parte B (2 puntos):** Análisis de correlación
1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
3. Interpreta la fuerza y dirección de la correlación

**Parte C (2 puntos):** Ajuste de distribución
1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
2. Reporta los parámetros ajustados (μ y σ)
3. Calcula la probabilidad de que un árbol seleccionado aleatoriamente de las parcelas control tenga biomasa > 150 kg
4. ¿Qué valor de biomasa representa el percentil 90 para los árboles de parcelas control?

In [12]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)  # Para reproducibilidad

# Parcelas control: bosque sin intervención
n_control = 35
dap_control = np.random.uniform(15, 50, n_control)  # DAP en cm
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)  # Asegurar valores positivos

# Parcelas AFIR: los árboles remanentes disponen de más recursos
n_afir = 30
dap_afir = np.random.uniform(18, 55, n_afir)  # DAP en cm
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

# Crear DataFrame
bosque_df = pd.DataFrame({
    'dap_cm': np.concatenate([dap_control, dap_afir]),
    'biomasa_kg': np.concatenate([biomasa_control, biomasa_afir]),
    'tratamiento': ['Control']*n_control + ['AFIR']*n_afir
})

print(bosque_df.head())
print(f"\nEspecies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)")

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control

Especies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)


In [15]:
# Parte A: Comparación de grupos de tratamiento

control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg']
afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg']

print("Estadisticas descriptivas - Control:")
print(f"Media: {control.mean():.2f}, Desv. estandar: {control.std():.2f}, Mediana: {control.median():.2f}")

print("\nEstadísticas descriptivas - AFIR:")
print(f"Media: {afir.mean():.2f}, Desv. estandar: {afir.std():.2f}, Mediana: {afir.median():.2f}")

# H0: no hay diferencia entre las medias de biomasa (mu_control = mu_afir)
# H1: si hay diferencia entre las medias de biomasa (mu_control != mu_afir)

t_stat, p_valor = stats.ttest_ind(control, afir)
print(f"\nEstadístico t: {t_stat:.4f}")
print(f"p-valor: {p_valor:.4f}")

alpha = 0.05
if p_valor < alpha:
    print(f"\nComo p-valor ({p_valor:.4f}) < alpha ({alpha}), se rechaza H0.")
    print("Conclusión: existe una diferencia estadisticamente significativa en la biomasa media entre Control y AFIR.")
else:
    print(f"\nComo p-valor ({p_valor:.4f}) >= alpha ({alpha}), no se rechaza H0.")
    print("Conclusión: no hay evidencia suficiente de una diferencia significativa en la biomasa media.")


Estadisticas descriptivas - Control:
Media: 269.64, Desv. estandar: 158.21, Mediana: 232.87

Estadísticas descriptivas - AFIR:
Media: 325.42, Desv. estandar: 213.52, Mediana: 239.12

Estadístico t: -1.2070
p-valor: 0.2319

Como p-valor (0.2319) >= alpha (0.05), no se rechaza H0.
Conclusión: no hay evidencia suficiente de una diferencia significativa en la biomasa media.


In [19]:
# Parte B: Análisis de correlación

r, p_corr = stats.pearsonr(bosque_df['dap_cm'], bosque_df['biomasa_kg'])
print(f"Coeficiente de correlación de Pearson: {r:.4f}")
print(f"p-valor: {p_corr:.6f}")

alpha = 0.05
if p_corr < alpha:
    print(f"\nComo p-valor < {alpha}, la correlación es estadisticamente significativa.")
else:
    print(f"\nComo p-valor >= {alpha}, la correlación no es estadisticamente significativa.")

print("\nInterpretación: la correlación es positiva y fuerte" if r > 0.7 else "\nInterpretación: la correlación es positiva y moderada" if r > 0.4 else "\nInterpretación: la correlación es debil")
print("Esto indica que a mayor DAP (diametro), mayor es la biomasa del árbol, lo cual es consistente con la relación alometrica esperada.")


Coeficiente de correlación de Pearson: 0.9565
p-valor: 0.000000

Como p-valor < 0.05, la correlación es estadisticamente significativa.

Interpretación: la correlación es positiva y fuerte
Esto indica que a mayor DAP (diametro), mayor es la biomasa del árbol, lo cual es consistente con la relación alometrica esperada.


In [21]:
# Parte C: Ajuste de distribución

mu, sigma = stats.norm.fit(control)
print(f"Parametros ajustados de la distribución normal (parcelas Control):")
print(f"mu = {mu:.2f}, sigma = {sigma:.2f}")

prob_mayor_150 = 1 - stats.norm.cdf(150, mu, sigma)
print(f"\nProbabilidad de que un árbol de parcela Control tenga biomasa > 150 kg: {prob_mayor_150:.4f}")

percentil_90 = stats.norm.ppf(0.90, mu, sigma)
print(f"\nValor de biomasa correspondiente al percentil 90: {percentil_90:.2f} kg")


Parametros ajustados de la distribución normal (parcelas Control):
mu = 269.64, sigma = 155.93

Probabilidad de que un árbol de parcela Control tenga biomasa > 150 kg: 0.7785

Valor de biomasa correspondiente al percentil 90: 469.48 kg


---
## Problema 3: Ajuste de Curva de Respuesta a la Luz (7 puntos)

La fotosíntesis depende de la intensidad de luz siguiendo una curva de saturación. La **hipérbola rectangular** se usa comúnmente para modelar esta relación:

$$A = \frac{A_{max} \cdot I}{K + I} - R_d$$

Donde:
- $A$ = tasa de fotosíntesis neta (μmol CO₂ m⁻² s⁻¹)
- $A_{max}$ = tasa máxima de fotosíntesis a saturación de luz
- $I$ = intensidad de luz (μmol fotones m⁻² s⁻¹, PAR)
- $K$ = constante de media saturación (nivel de luz en el que A = A_max/2 - R_d)
- $R_d$ = tasa de respiración en oscuridad (CO₂ liberado cuando I = 0)

Los datos provienen de mediciones de *Cecropia sp.* ("cetico"), una especie pionera característica de la Amazonía peruana, muy importante en la regeneración de bosques perturbados.

### Tus Tareas:

**Parte A (2 puntos):** Define el modelo y la función de costo
1. Escribe una función `respuesta_luz(I, Amax, K, Rd)` que implemente la ecuación anterior
2. Escribe una función de costo `respuesta_luz_mse(params, I_datos, A_datos)` que calcule el error cuadrático medio entre las tasas de fotosíntesis observadas y predichas
3. Prueba tu función `respuesta_luz` calculando A para I = 500 con Amax=25, K=200, Rd=2

**Parte B (3 puntos):** Ajusta el modelo usando optimización
1. Usa `scipy.optimize.minimize` para encontrar los parámetros óptimos (Amax, K, Rd) que minimicen el MSE
2. Usa valores iniciales: Amax=20, K=150, Rd=1
3. Reporta los parámetros ajustados y el MSE final
4. Ajusta también el modelo usando `scipy.optimize.curve_fit` y compara los resultados

**Parte C (2 puntos):** Evalúa e interpreta el modelo
1. Calcula los valores de fotosíntesis predichos usando tus parámetros ajustados
2. Calcula R² (coeficiente de determinación) para evaluar el ajuste del modelo:
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
3. Calcula el **punto de compensación lumínico** (el nivel de luz donde A = 0, es decir, la fotosíntesis iguala a la respiración). Pista: despeja I cuando A = 0
4. ¿Cuál es la tasa de fotosíntesis a saturación lumínica (Amax - Rd)?

In [22]:
# Datos de curva de respuesta a la luz - Cecropia sp. (cetico)
# Mediciones en parcela de investigación, Madre de Dios

# PAR (radiación fotosintéticamente activa) en μmol fotones m⁻² s⁻¹
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])

# Tasa de fotosíntesis neta en μmol CO₂ m⁻² s⁻¹
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

print(f"Rango PAR: {par_datos.min()} a {par_datos.max()} μmol fotones m⁻² s⁻¹")
print(f"Rango fotosíntesis: {foto_datos.min()} a {foto_datos.max()} μmol CO₂ m⁻² s⁻¹")
print("Especie: Cecropia sp. (cetico) - pionera amazónica")

Rango PAR: 0 a 1800 μmol fotones m⁻² s⁻¹
Rango fotosíntesis: -1.8 a 23.1 μmol CO₂ m⁻² s⁻¹
Especie: Cecropia sp. (cetico) - pionera amazónica


In [23]:
# Parte A: Define el modelo y la función de costo

def respuesta_luz(I, Amax, K, Rd):
    return (Amax * I) / (K + I) - Rd

def respuesta_luz_mse(params, I_datos, A_datos):
    Amax, K, Rd = params
    A_predicho = respuesta_luz(I_datos, Amax, K, Rd)
    return np.mean((A_datos - A_predicho)**2)

A_prueba = respuesta_luz(500, 25, 200, 2)
print(f"A para I=500, Amax=25, K=200, Rd=2: {A_prueba:.4f}")

A para I=500, Amax=25, K=200, Rd=2: 15.8571


In [25]:
# Parte B: Ajusta el modelo usando optimización

params_iniciales = [20, 150, 1]

resultado = minimize(respuesta_luz_mse, params_iniciales, args=(par_datos, foto_datos))
Amax_opt, K_opt, Rd_opt = resultado.x

print("Resultados con scipy.optimize.minimize:")
print(f"Amax = {Amax_opt:.4f}, K = {K_opt:.4f}, Rd = {Rd_opt:.4f}")
print(f"MSE final: {resultado.fun:.4f}")

popt, pcov = curve_fit(respuesta_luz, par_datos, foto_datos, p0=params_iniciales)
print("\nResultados con scipy.optimize.curve_fit:")
print(f"Amax = {popt[0]:.4f}, K = {popt[1]:.4f}, Rd = {popt[2]:.4f}")

print("\nComparación: ambos metodos deberian dar parametros muy similares, ya que minimize (con MSE) y curve_fit (con minimos cuadrados) resuelven esencialmente el mismo problema de optimización.")


Resultados con scipy.optimize.minimize:
Amax = 28.4461, K = 134.7545, Rd = 2.6272
MSE final: 0.2373

Resultados con scipy.optimize.curve_fit:
Amax = 28.4460, K = 134.7545, Rd = 2.6272

Comparación: ambos metodos deberian dar parametros muy similares, ya que minimize (con MSE) y curve_fit (con minimos cuadrados) resuelven esencialmente el mismo problema de optimización.


In [27]:
# Parte C: Evalúa e interpreta el modelo

Amax, K, Rd = popt

foto_predicho = respuesta_luz(par_datos, Amax, K, Rd)

ss_res = np.sum((foto_datos - foto_predicho)**2)
ss_tot = np.sum((foto_datos - np.mean(foto_datos))**2)
r2 = 1 - ss_res / ss_tot
print(f"R^2 del modelo ajustado: {r2:.4f}")

# Punto de compensacion luminico: A = 0 => Amax*I/(K+I) - Rd = 0 => I = Rd*K/(Amax - Rd)
punto_compensacion = Rd * K / (Amax - Rd)
print(f"\nPunto de compensación luminico: {punto_compensacion:.2f} umol fotones m^-2 s^-1")

tasa_saturacion = Amax - Rd
print(f"\nTasa de fotosintesis a saturación luminica (Amax - Rd): {tasa_saturacion:.2f} umol CO2 m^-2 s^-1")


R^2 del modelo ajustado: 0.9966

Punto de compensación luminico: 13.71 umol fotones m^-2 s^-1

Tasa de fotosintesis a saturación luminica (Amax - Rd): 25.82 umol CO2 m^-2 s^-1


---
## Lista de verificación para entrega

Antes de entregar, verifica que:

- [ ] Todas las celdas de código se ejecutan sin errores
- [ ] Los tres problemas están completos
- [ ] Los resultados son visibles en todas las celdas
- [ ] Tu nombre está incluido al inicio
- [ ] El archivo está guardado en la carpeta `tareas` de tu repositorio privado de GitHub
- [ ] El archivo está subido a GitHub **antes del plazo**
- [ ] El enlace a tu notebook está enviado en Canvas